In [1]:
import datetime
import sys
# path为tools的上层文件夹
sys.path.append('D:\milimili')
import numpy as np
import pandas as pd
from configs import TE_app_token
from tools.数数查询 import get_sqldata
token=TE_app_token['Doomsday Vanguard']

### 不同来源渠道来源的用户是否有时长差异

In [27]:
sql = """
-- 正式关卡时长
-- 每日挑战
-- 每日活动
-- 宠物副本
-- 芯片副本
-- 试炼
-- 活动副本
-- 公会讨伐战
-- 事件发生时间：24-01-01 至今 24-04-07
-- 用户来源时间：24-01-01 - 24-01-31
with tab as (
select * from (
select "#user_id" as "id1","#os","#country_code","$part_event","$part_date","level_time"
from ta.v_event_33 
where "$part_date" between '2024-01-01' and '2024-04-07'
and "#os" != 'HarmonyOS'
and "$part_event" in (
'Game_Level_Finish',
'DailyChallenge_level_finish',
'DailyActivity_level_finish',
'DailyPetChallenge_level_finish',
'ChipChallenge_level_finish',
'TrialLevel_Finish',
'GuildChallenge_level_finish',
'ActivityChallenge_level_finish'
)
) a 
join 
-- 用户属性 来源时间、累计付费金额、累计广告次数、当前关卡数、来源渠道、国家
(
select "#user_id" ,"#active_time","max_chapter_id","total_pay_amount","adjust_country_code","#adjust_network_name"
from ta.v_user_33 
where "adjust_country_code" != 'cn'
and to_char("#active_time",'yyyy-mm-dd') between  '2024-01-01' and '2024-01-31'
)  b  on a."id1" = b."#user_id")
select "#os","#adjust_network_name" ,"$part_event","$part_date",sum("level_time"),count(distinct "id1")
from tab   
group by "#os","#adjust_network_name","$part_event","$part_date"
"""

In [28]:
res_network=get_sqldata(token,sql)

In [29]:
res_network.columns = ['os','渠道','事件','日期','时长','用户数',]

In [30]:
res_network.head()

,os,渠道,事件,日期,时长,用户数
0,iOS,Organic,DailyPetChallenge_level_finish,2024-01-12,125366.0,617
1,iOS,Applovin,DailyPetChallenge_level_finish,2024-01-09,124117.0,595
2,Android,Unattributed,DailyPetChallenge_level_finish,2024-01-09,7932.0,38
3,iOS,Organic,DailyPetChallenge_level_finish,2024-01-08,77664.0,367
4,Android,Google Ads ACI,DailyPetChallenge_level_finish,2024-01-08,61466.0,289


In [31]:
res_network=res_network.query('渠道==渠道')

In [32]:
res_network['时长']=res_network['时长'].astype('float')
res_network['用户数']=res_network['用户数'].astype('float')

In [33]:
res_network['人均时长']=res_network['时长']/res_network['用户数']/60

In [59]:
res_network_gp=res_network.groupby(by=['os','渠道','事件'])['人均时长'].mean().reset_index()

In [60]:
res_network_gp.to_excel('渠道.xlsx')

### 不同付费金额的用户，在不同事件的时长分布

In [42]:
sql = """
-- 正式关卡时长
-- 每日挑战
-- 每日活动
-- 宠物副本
-- 芯片副本
-- 试炼
-- 活动副本
-- 公会讨伐战
-- 事件时间：24-01-01 至今 24-04-07
with tab as (
select * from (
select "#user_id" as "id1","#os","#country_code","$part_event","$part_date","level_time"
from ta.v_event_33 
where "$part_date" between '2024-01-01' and '2024-04-07'
and "#os" != 'HarmonyOS'
and "$part_event" in (
'Game_Level_Finish',
'DailyChallenge_level_finish',
'DailyActivity_level_finish',
'DailyPetChallenge_level_finish',
'ChipChallenge_level_finish',
'TrialLevel_Finish',
'GuildChallenge_level_finish',
'ActivityChallenge_level_finish'
)
) a 
join 
-- 用户属性 来源时间、累计付费金额、累计广告次数、当前关卡数、来源渠道、国家
(
select "#user_id" ,"#active_time","max_chapter_id",
case when "total_pay_amount" is NULL  then '0氪玩家'
     when "total_pay_amount" < 20  then '微氪玩家'
     when "total_pay_amount" < 100  then '轻氪玩家'
     when "total_pay_amount" < 300  then '中氪玩家'
     when "total_pay_amount" < 1000  then '大R玩家'
     else '重度玩家'
end  as "付费区间",
"total_pay_amount","adjust_country_code","#adjust_network_name"
from ta.v_user_33 
where "adjust_country_code" != 'cn'
and to_char("#active_time",'yyyy-mm-dd') between  '2024-01-01' and '2024-01-31'
)  b  on a."id1" = b."#user_id")
select "#os","#adjust_network_name" ,"付费区间",
"$part_event","$part_date",
sum("level_time"),count(distinct "id1")
from tab   
group by "#os","#adjust_network_name","付费区间","$part_event","$part_date"

"""

In [43]:
res_pay_range = get_sqldata(token,sql)

In [44]:
res_pay_range.columns= ['os','渠道','付费区间','事件','日期','时长','人数']

In [45]:
res_pay_range.head()

,os,渠道,付费区间,事件,日期,时长,人数
0,iOS,Organic,0氪玩家,ChipChallenge_level_finish,2024-01-14,17640.0,90
1,iOS,Organic,微氪玩家,ChipChallenge_level_finish,2024-01-14,16020.0,78
2,iOS,Amoad-DH,微氪玩家,ChipChallenge_level_finish,2024-01-14,1080.0,6
3,iOS,Applovin,微氪玩家,ChipChallenge_level_finish,2024-01-11,10620.0,53
4,Android,Unattributed,中氪玩家,ChipChallenge_level_finish,2024-01-11,360.0,2


In [46]:
res_pay_range=res_pay_range.query('渠道==渠道')
res_pay_range['时长']=res_pay_range['时长'].astype('float')
res_pay_range['人数']=res_pay_range['人数'].astype('float')
res_pay_range['人均时长']=res_pay_range['时长']/res_pay_range['人数']/60

In [57]:
res_pay_range_gp = res_pay_range.groupby(by=['os','渠道','付费区间','事件'])['人均时长'].mean().reset_index()

In [58]:
res_pay_range_gp.to_excel('付费金额与时长.xlsx')

### 用户进入游戏后，各个板块的时长随生命周期变化

In [49]:
sql = """
-- 正式关卡时长
-- 每日挑战
-- 每日活动
-- 宠物副本
-- 芯片副本
-- 试炼
-- 活动副本
-- 公会讨伐战
-- 用户：24-01-01 至今 24-04-07
with tab as (
select  a."id1",a."$part_event",a."$part_date",a."level_time",date_diff('day',b."#active_time",a."#event_time") as "live_day"
from (
select "#user_id" as "id1","#os","#country_code","$part_event","$part_date","#event_time","level_time"
from ta.v_event_33 
where "$part_date" between '2024-01-01' and '2024-04-07'
and "#os" != 'HarmonyOS'
and "$part_event" in (
'Game_Level_Finish',
'DailyChallenge_level_finish',
'DailyActivity_level_finish',
'DailyPetChallenge_level_finish',
'ChipChallenge_level_finish',
'TrialLevel_Finish',
'GuildChallenge_level_finish',
'ActivityChallenge_level_finish'
)
) a 
join 
-- 用户属性 来源时间、累计付费金额、累计广告次数、当前关卡数、来源渠道、国家
(
select "#user_id" ,"#active_time",
case when "total_pay_amount" is NULL  then '0氪玩家'
     when "total_pay_amount" < 20  then '微氪玩家'
     when "total_pay_amount" < 100  then '轻氪玩家'
     when "total_pay_amount" < 300  then '中氪玩家'
     when "total_pay_amount" < 1000  then '大R玩家'
     else '重度玩家'
end  as "付费区间",
"total_pay_amount","adjust_country_code","#adjust_network_name"
from ta.v_user_33 
where "adjust_country_code" != 'cn'
and to_char("#active_time",'yyyy-mm-dd') between  '2024-01-01' and '2024-01-31'
)  b  on a."id1" = b."#user_id")
select "$part_event","live_day",sum("level_time"),count(DISTINCT "id1")  from tab
group by "$part_event","live_day"
order by  "$part_event","live_day"

"""

In [50]:
res_live = get_sqldata(token,sql)

In [51]:
res_live.columns = ['事件','生命周期天数','时长','人数']

In [53]:
res_live['时长']=res_live['时长'].astype('float')
res_live['人数']=res_live['人数'].astype('float')
res_live['人均时长']=res_live['时长']/res_live['人数']/60

In [54]:
res_live.to_excel('生命周期与时长.xlsx',index=False)